In [1]:
!pip install xgboost

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix

from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [3]:
df = pd.read_csv("/content/DWT_FDispersionEntropy_Features.csv")

X = df.loc[:, "FDispEn_D1":"FDispEn_D5"].values
y = df["Label"].values

y = y - 1


In [4]:
models = {
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", C=1, gamma="scale"))
    ]),

    "MLP": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", MLPClassifier(
            hidden_layer_sizes=(100,),
            max_iter=500,
            random_state=42
        ))
    ]),

    "XGBoost": XGBClassifier(
        objective="multi:softmax",
        num_class=len(np.unique(y)),
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=42,
        eval_metric="mlogloss"
    )
}

In [5]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [6]:
def sensitivity_specificity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)

    TP = np.diag(cm)
    FN = np.sum(cm, axis=1) - TP
    FP = np.sum(cm, axis=0) - TP
    TN = np.sum(cm) - (TP + FP + FN)

    sensitivity = np.mean(TP / (TP + FN + 1e-9))
    specificity = np.mean(TN / (TN + FP + 1e-9))
    score = (sensitivity + specificity) / 2

    return sensitivity, specificity, score

In [7]:
def sensitivity_specificity_micro(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)

    TP = np.trace(cm)
    total = np.sum(cm)
    FP = total - TP
    FN = FP

    # global one-vs-rest (operasional)
    num_classes = cm.shape[0]
    TN = total * (num_classes - 1) - FP

    sensitivity = TP / (TP + FN + 1e-9)     # = micro recall
    specificity = TN / (TN + FP + 1e-9)
    score = (sensitivity + specificity) / 2

    return sensitivity, specificity, score

In [9]:
results_micro = {}

for name, model in models.items():
    accs, precs, recs, f1s = [], [], [], []
    sens, specs, scores = [], [], []

    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        accs.append(accuracy_score(y_test, y_pred))
        precs.append(precision_score(y_test, y_pred, average="micro"))
        recs.append(recall_score(y_test, y_pred, average="micro"))
        f1s.append(f1_score(y_test, y_pred, average="micro"))

        s, sp, sc = sensitivity_specificity_micro(y_test, y_pred)
        sens.append(s)
        specs.append(sp)
        scores.append(sc)

    results_micro[name] = {
        "Accuracy": np.mean(accs),
        "Precision": np.mean(precs),
        "Sensitivity": np.mean(sens),
        "Specificity": np.mean(specs),
        "Score": np.mean(scores),
        "F1-Score": np.mean(f1s)
    }

results_micro_df = pd.DataFrame(results_micro).T
results_micro_df


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptro

,Accuracy,Precision,Sensitivity,Specificity,Score,F1-Score
SVM,0.959474,0.959474,0.959474,0.989868,0.974671,0.959474
MLP,0.948947,0.948947,0.948947,0.987237,0.968092,0.948947
XGBoost,0.908947,0.908947,0.908947,0.977237,0.943092,0.908947
